In [ ]:
import os
from dotenv import load_dotenv
from uuid import uuid4
from langchain_community.document_loaders import PyPDFLoader
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings

COLLECTION_NAME = "company-policy-rag"

# 1.Embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


# 2.Chat model
model = ChatOllama(model="qwen3")

# 3. Load PDF
loader = PyPDFLoader(r"D:\RAG\VectotDatabases\Qdrant\doc\llama2-research-paper.pdf")
pages = loader.load()
print("PDF pages loaded:", len(pages))

# 4. Split documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=120,
)
chunks = text_splitter.split_documents(pages)
print("Chunks created:", len(chunks))


# 5. Add useful metadata
for chunk_index, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = chunk_index
    chunk.metadata["filename"] = "llama2-research-paper.pdf"


# Load the .env file
load_dotenv()

qdrant_api_key = os.getenv("QDRANT_API_KEY")
qdrant_cluster_endpoint = os.getenv("QDRANT_Cluster_Endpoint")


client = QdrantClient(api_key=qdrant_api_key, url=qdrant_cluster_endpoint)

# 6. Create collection
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=384,
            distance=models.Distance.COSINE,
        ),
    )

